# ThinkPRM — Week 1-2 kickoff (Kaggle)

Setup: **Accelerator = GPU T4 x2**, **Internet = ON**.
Attach this repo as a Kaggle Dataset (or `git clone` it), then run cells top to bottom.
Each section ends with a checkpoint. Total budget: ~15-20 GPU hours.

In [ ]:
# 0. Environment — repo must be at /kaggle/input/thinkprm-repo or ./thinkprm
import os, shutil
REPO = "/kaggle/input/thinkprm-repo" if os.path.exists("/kaggle/input/thinkprm-repo") else "./thinkprm"
os.chdir(REPO)
print("repo:", os.getcwd())
!pip install -q -r requirements.txt
!python -c "import torch; print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0))"

In [ ]:
# 1. Data pipeline (~10 min, mostly download)
!python scripts/data/prepare_prm800k.py --out data/prm800k_unified.jsonl
!python scripts/data/prepare_mathshepherd.py --out data/mathshepherd_unified.jsonl --max_records 20000

✅ **Checkpoint 1**: `gold_0` ≈ 40% of PRM800K steps (the negatives rejection sampling throws away); Math-Shepherd `parse_failed` < 1%.

In [ ]:
# 2. Sanity-check the verifier format + reward parser (CPU, instant)
import sys; sys.path.insert(0, "scripts/train")
from rewards import build_verifier_prompt, parse_verification
p = build_verifier_prompt("What is 2+2?", ["2+2 = 4.", "So the answer is 5."])
print(p)
demo = "Step 1: correct\nStep 2: incorrect\nFirst error step: 2\nFinal: incorrect\nConfidence: 0.82"
print(parse_verification(demo))
assert parse_verification("garbage") is None
print("parser OK")

In [ ]:
# 3. DiscPRM baseline — cheap, fits T4 (~4-6 GPU hours)
!python scripts/train/train_discprm.py --config configs/discprm_1p5b.yaml

✅ **Checkpoint 2**: loss decreases, no NaN. (Step-level accuracy on phase1_test should be clearly above chance.)

In [ ]:
# 4. SFT baseline = rejection-sampling proxy (~4-6 GPU hours)
!python scripts/train/train_sft.py --config configs/sft_1p5b.yaml

In [ ]:
# 5. Evaluate SFT: accuracy + ECE + Brier on phase1_test (200 items, ~30 min)
!python scripts/evaluation/evaluate.py --model outputs/sft-1p5b --data data/prm800k_unified.jsonl --split phase1_test --max_n 200

✅ **Checkpoint 3**: record **step accuracy / solution accuracy / ECE / Brier** — this is the baseline Part A must beat.

Next: GRPO pilot (short sequences) → see `docs/WEEK1-2.md` Day 6-7. Full GRPO needs an A6000 / WashU cluster.